In [1]:
#Importação de bibliotecas
import polars as pl

In [ ]:
# Leitura do arquivo e análise

In [16]:
df = pl.read_csv("/home/ulisses/Desafio_01_FTIA/Dados/dataset.csv")

In [17]:
display(df)

idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
i64,str,str,f64,i64,i64,f64,i64,str,i64,i64
52,"""MS""","""Ensino Superior""",2911.82,14,690,5934.19,60,"""Viagem""",1,0
49,"""RR""","""Ensino Médio""",3176.43,5,676,6791.38,24,"""Outros""",2,0
48,"""AL""","""Ensino Superior""",3429.4,23,558,14666.78,24,"""Outros""",5,1
59,"""GO""","""Ensino Fundamental""",10096.7,2,739,19053.24,24,"""Reforma""",0,0
37,"""RJ""","""Ensino Fundamental""",3033.25,24,797,6171.31,6,"""Educação""",0,1
…,…,…,…,…,…,…,…,…,…,…
31,"""PB""","""Ensino Médio""",3374.04,8,677,10198.45,24,"""Outros""",2,0
64,"""AM""","""Ensino Superior""",3638.91,20,643,15374.4,24,"""Reforma""",1,0
48,"""MT""","""Ensino Médio""",2211.31,21,486,8236.1,48,"""Saúde""",0,1


In [22]:
# Analisar a completude dos dados - Aparentemente todos os dados estão presentes
df_com = df.null_count()
display(df_com)

idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0


In [24]:
# Análise de qualidade dos dados
df.describe()

statistic,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
str,f64,str,str,f64,f64,f64,f64,f64,str,f64,f64
"""count""",5000.0,"""5000""","""5000""",5000.0,5000.0,5000.0,5000.0,5000.0,"""5000""",5000.0,5000.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0
"""mean""",47.7674,null,null,5228.633582,12.6622,652.4812,14264.657378,29.922,null,1.9004,0.344
"""std""",31.646099,null,null,3968.478198,7.517539,84.817076,8268.261465,15.482515,null,1.864719,0.475089
"""min""",18.0,""" AC ""","""Ensino Fundamental""",100.0,1.0,335.0,1721.48,6.0,"""Educação""",0.0,0.0
"""25%""",32.0,null,null,2697.41,6.0,594.0,8467.48,18.0,null,0.0,0.0
"""50%""",46.0,null,null,4146.73,13.0,653.0,12367.3,24.0,null,1.0,0.0
"""75%""",61.0,null,null,6458.74,19.0,710.0,17763.21,36.0,null,3.0,1.0
"""max""",750.0,"""TO""","""Pós-graduação""",44814.19,74.0,850.0,50000.0,60.0,"""Viagem""",7.0,1.0


In [30]:
# Inconsistências:
## Idade máxima: 750 - Analisar outras idades que possam estar fora da realidade (abaixo de 18?) - Não possui.
## Tempo_emprego: max 74, identificar se está consistente com a idade (menor idade para contribuição é 14 anos - menor aprendiz)
## Score_credito: Verificar como funciona.No Brasil(Serasa) - o score vai de 0 a 1000. Apresenta aderência.

df_incons = df.filter(
# Identifica pessoas acima de 115 (pessoa mais idosa do Brasil)
    (pl.col("idade") > 115) |
# Verifica se o tempo de emprego é compatível com a idade
    (pl.col("idade") - pl.col("tempo_emprego_anos")   <= 14)
)
display(df_incons)
print(df_incons.count())

idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
i64,str,str,f64,i64,i64,f64,i64,str,i64,i64
37,"""RJ""","""Ensino Fundamental""",3033.25,24,797,6171.31,6,"""Educação""",0,1
22,"""CE""","""Ensino Superior""",2571.1,19,708,19958.21,24,"""Educação""",0,0
19,"""GO""","""Ensino Superior""",5212.21,14,787,12724.96,60,"""Outros""",2,0
20,"""MA""","""Ensino Médio""",4557.43,14,700,11019.03,48,"""Saúde""",2,0
32,"""AC""","""Ensino Superior""",3843.02,19,653,13189.32,12,"""Educação""",0,0
…,…,…,…,…,…,…,…,…,…,…
22,"""ES""","""Pós-graduação""",6896.04,9,670,14203.9,12,"""Reforma""",1,0
34,"""SC""","""Pós-graduação""",2640.66,25,801,15602.87,60,"""Reforma""",1,0
29,"""DF""","""Ensino Médio""",3200.64,25,711,13543.79,6,"""Veículo""",0,1


shape: (1, 11)
┌───────┬────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ idade ┆ estado ┆ escolarida ┆ renda_mens ┆ … ┆ prazo_mese ┆ finalidade ┆ dividas_at ┆ inadimplen │
│ ---   ┆ ---    ┆ de         ┆ al         ┆   ┆ s          ┆ ---        ┆ ivas       ┆ te         │
│ u32   ┆ u32    ┆ ---        ┆ ---        ┆   ┆ ---        ┆ u32        ┆ ---        ┆ ---        │
│       ┆        ┆ u32        ┆ u32        ┆   ┆ u32        ┆            ┆ u32        ┆ u32        │
╞═══════╪════════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 778   ┆ 778    ┆ 778        ┆ 778        ┆ … ┆ 778        ┆ 778        ┆ 778        ┆ 778        │
└───────┴────────┴────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘


In [31]:
# Me ocorreu que os estados e a escolaridade (colunas texto) podem ter valores inválidos mesmo que não estejam nulos.
## Verificar:
print(df["estado"].unique())
print(df["escolaridade"].unique())
print(df["finalidade"].unique())


shape: (85,)
Series: 'estado' [str]
[
	"SC"
	"ES"
	"SE"
	"RR"
	" TO "
	…
	"  PB  "
	"  RS  "
	" PA "
	" RJ "
	"  RS "
]
shape: (4,)
Series: 'escolaridade' [str]
[
	"Ensino Superior"
	"Pós-graduação"
	"Ensino Médio"
	"Ensino Fundamental"
]
shape: (6,)
Series: 'finalidade' [str]
[
	"Saúde"
	"Educação"
	"Veículo"
	"Viagem"
	"Outros"
	"Reforma"
]


O retorno da lista única para Escolaridade e Finalidade não apresentam à primeira vista nenhuma inconsistência.

Escolaridade: "Ensino Superior", "Pós-graduação", "Ensino Médio", "Ensino Fundamental"
Finalidade: "Saúde", "Educação", "Veículo", "Viagem", "Outros", "Reforma"

In [41]:
# Ao realizar a analise de estados únicos recebi 85 possibilidades, inicialmente encontrei espaços na grafia das siglas.
## Vou realizar a limpeza antes de proceder com a análise
df = df.with_columns(
    pl.col("estado").str.strip_chars()
)

print(df["estado"].unique())

shape: (27,)
Series: 'estado' [str]
[
	"AP"
	"AL"
	"PE"
	"MT"
	"AM"
	…
	"RS"
	"RJ"
	"RR"
	"DF"
	"PR"
]


In [46]:
# A quantidade de siglas únicas agora está consistente com a quantidade de estados no Brasil - 27
df_estado = df["estado"].unique()
print(df_estado.count())

27


In [48]:
# verificando se apenas as siglas corretas foram utilizadas
estados_validos = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", 
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", 
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]
estados_invalidos = df.filter(
    ~pl.col("estado").is_in(estados_validos)
)

display(estados_invalidos)

idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
i64,str,str,f64,i64,i64,f64,i64,str,i64,i64


O retorno vazio do df acima significa que as siglas estão de acordo com as siglas corretas.

Análise para correção das inconsitências encontradas para idade e tempo_emprego_anos

- Idade acima de 115 (considerando a idade da pessoa mais idosa do Brasil)
- Pessoas com tempo de emprego incompátivel com a idade. Atualmente no Brasil idade mínima para assinatura da carteira é 14 anos (Menor aprendiz).
